In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
import ipywidgets
import plotly.graph_objects as go
import rasterio
from matplotlib.colors import LightSource
from ipywidgets import interact, FloatSlider, IntSlider
import plotly.io as pio
from scipy.spatial import Delaunay
import pyvista as pv
import rasterio
from scipy.interpolate import RegularGridInterpolator
from plotly.subplots import make_subplots
from rasterio.enums import Resampling
from rasterio.windows import Window
pio.renderers.default = 'vscode'
from utils import *
import trimesh
import pyvista as pv
pv.set_jupyter_backend('trame')

In [ ]:
file_path = "/Users/pierazhi/Desktop/tifs/LDEM_20M.tif" # Loads .tif
dimension_scene, _ = read_tif(file_path) # Gets dimension of the scene
#visualize_tif(file_path, np.floor(dimension_scene / 4)) # Shows half the scene (cause it is big)

In [ ]:
file_cut = '/Users/pierazhi/Desktop/tifs/LDEM_20M_clip.tif' # Creates the cut file's path
cut_pixels = 150 # Choose the square, starting from the center, that is going to be cut from the whole scene

# write_and_save_tif(file_path, file_cut, cut_pixels, 47, 43) # Saves the cut scene
# write_and_save_tif(file_path, file_cut, cut_pixels, 0, 0) # Saves the cut scene
write_and_save_tif(file_path, file_cut, cut_pixels, -25, 25) # Saves the cut scene

dimension_scene_cut, _ = read_tif(file_cut) # Read the cut scene
visualize_tif(file_cut, dimension_scene_cut) # Viualizes it

In [ ]:
passo = int(cut_pixels)
X, Y, Z, triangoli, Xe, Ye, Ze, normali, baricentri, Xg, Yg, Zg = tif2mesh(file_cut, np.floor(dimension_scene_cut*3), passo) # Creates the mesh of triangles from the starting .tif

nuova_risoluzione_metri = 20
file_cut_up = f'/Users/pierazhi/Desktop/tifs/LDEM_{nuova_risoluzione_metri}M_clip_up.tif' # Creates file's path for the cut and upsampled scene 
passo_up = cut_pixels * 2
upsampling_geotiff(file_cut, file_cut_up, nuova_risoluzione_metri, metodo_interpolazione='cubic') # Upsamples the cut file to a new resolution
X_up, Y_up, Z_up, triangoli_up, Xe_up, Ye_up, Ze_up, normali_up, baricentri_up, Xg_up, Yg_up, Zg_up  = tif2mesh(file_cut_up, np.floor(dimension_scene_cut*3), passo_up) # Creates the mesh of triangles from the starting .tif

In [ ]:
show_normals = False
show_triangles = False

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=(
        f"Mesh Originale (20m) - Triangoli: {len(triangoli):,}", 
        f"Mesh Interpolata {nuova_risoluzione_metri} m - Triangoli: {len(triangoli_up):,}"
    )
)
fig.add_trace(go.Mesh3d(
    x=X, y=Y, z=Z, i=triangoli[:, 0], j=triangoli[:, 1], k=triangoli[:, 2],
    intensity=Z, colorscale='Earth_r', showscale=False, opacity=0.85, name="Orig 20m"
), row=1, col=1)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe, y=Ye, z=Ze, mode='lines',
        line=dict(color='rgb(30,30,30)', width=1), showlegend=False
    ), row=1, col=1)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg, y=Yg, z=Zg, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=1)

fig.add_trace(go.Mesh3d(
    x=X_up, y=Y_up, z=Z_up, i=triangoli_up[:, 0], j=triangoli_up[:, 1], k=triangoli_up[:, 2],
    intensity=Z_up, colorscale='Earth_r', showscale=True, opacity=0.85,
    colorbar=dict(title="Quota (m)", x=1.05), name="Interp 10m"
), row=1, col=2)

if show_triangles:
    fig.add_trace(go.Scatter3d(
        x=Xe_up, y=Ye_up, z=Ze_up, mode='lines',
        line=dict(color='rgb(50,50,50)', width=0.8), showlegend=False
    ), row=1, col=2)

if show_normals:
    fig.add_trace(go.Scatter3d(
        x=Xg_up, y=Yg_up, z=Zg_up, mode='lines',
        line=dict(color='rgb(255, 0, 0)', width=3), name="Vettori Normali", showlegend=False
    ), row=1, col=2)

scene_config = dict(
    xaxis_title="X (Metri)",
    yaxis_title="Y (Metri)",
    zaxis_title="Z (Metri)",
    aspectmode='data' 
    # aspectmode='manual',
    # aspectratio=dict(x=1, y=1, z=0.4)
)

fig.update_layout(
    scene=scene_config,   # Configura la vista di sinistra (scena 1)
    scene2=scene_config,  # Configura la vista di destra (scena 2)
    margin=dict(l=0, r=50, b=0, t=110),
    width=1150,           # Allargato leggermente per fare spazio alla barra dei colori
    height=750
)
fig.show()


In [ ]:
# --- 1. CONFIGURAZIONE MESH ---
vertices = np.vstack((X_up, Y_up, Z_up)).T
mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli_up)
mesh_trimesh.fix_normals()

faces_pv = np.hstack(np.pad(triangoli_up, ((0, 0), (1, 0)), constant_values=3))
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

# --- 2. CONFIGURAZIONE INIZIALE FINESTRA E TERRENO ---
plotter = pv.Plotter()
# plotter.add_mesh(mesh_pv, scalars=mesh_pv.points[:, 2], cmap="terrain", lighting=True, opacity=0.8)
plotter.add_mesh(
    mesh_pv, 
    color="darkgray",     # Colore grigio uniforme per tutta la superficie
    lighting=True,        # Mantiene le ombreggiature per far risaltare i rilievi 3D
    opacity=0.8
)
# Calcolo dei confini del terreno (matrice 2x3)
xmin, ymin, zmin = mesh_trimesh.bounds[0]
xmax, ymax, zmax = mesh_trimesh.bounds[1]
offset_pos = 1000  

# Dizionario di stato UNIFICATO per tutti i parametri modificabili
stato_parametri = {
    "xx": np.mean(X_up) - 100,
    "yy": np.mean(Y_up) - 100,
    "zz": np.max(Z_up) + 200,
    "azimut": 135.0,
    "elevazione": -45.0
}

# Lista globale dinamica per tracciare gli attori grafici e l'attore del testo
attori_raggi = []
attore_testo = None  # Tiene traccia del pannello di testo a schermo

MAX_BOUNCES = 4
colori_raggi = ["red", "yellow", "magenta", "black"]

# --- 3. FUNZIONE DI AGGIORNAMENTO GEOMETRIA E TESTO (CALLBACK) ---
def aggiorna_ray_tracing(*args):
    """Ricalcola ricorsivamente fino a 4 rimbalzi e aggiorna il testo dinamico a schermo."""
    global attori_raggi, attore_testo
    
    # 1. Rimuove tutti i vettori e i punti della passata precedente dalla GPU
    for attore in attori_raggi:
        plotter.remove_actor(attore)
    attori_raggi.clear()
    
    # Rimuove il vecchio blocco di testo sovrimpresso se esiste
    if attore_testo is not None:
        plotter.remove_actor(attore_testo)

    # 2. Inizializzazione variabili leggendo i dati aggiornati dagli slider
    origine_corrente = np.array([stato_parametri["xx"], stato_parametri["yy"], stato_parametri["zz"]])
    direzione_corrente = calcola_direzione_raggio(stato_parametri["azimut"], stato_parametri["elevazione"])
    
    # Costruiamo la stringa informativa da mostrare direttamente sullo schermo 3D
    stringa_log_schermo = (
        f"SORGENTE TX\n"
        f"Pos: [{stato_parametri['xx']:.1f}, {stato_parametri['yy']:.1f}, {stato_parametri['zz']:.1f}]\n"
        f"Angoli: Az={stato_parametri['azimut']:.1f}°, El={stato_parametri['elevazione']:.1f}°\n"
        f"--------------------------------------------------\n"
    )

    # Disegna la sorgente iniziale (TX) come una sfera blu
    tx_sfera = pv.Sphere(radius=25.0, center=origine_corrente)
    attori_raggi.append(plotter.add_mesh(tx_sfera, color="blue", lighting=True))

    # 3. Esecuzione dei rimbalzi (fino a un massimo di 4)
    for bounce in range(MAX_BOUNCES):
        p_end = origine_corrente + (direzione_corrente * 10000.0)
        p_intersezioni, id_facce = mesh_pv.ray_trace(origine_corrente, p_end)

        if len(p_intersezioni) > 0:
            # Ordinamento spaziale per distanza
            distanze = np.linalg.norm(p_intersezioni - origine_corrente, axis=1)
            idx_vicino = np.argmin(distanze)
            
            punto_contatto = p_intersezioni[idx_vicino, :]
            
            # Estrazione sicura dell'indice triangolo
            if hasattr(id_facce, "__len__"):
                if len(id_facce) > 0:
                    idx_faccia = int(id_facce[idx_vicino])
                else:
                    idx_faccia = int(id_facce)
            else:
                idx_faccia = int(id_facce)

            # Aggiunge i dati del rimbalzo corrente alla stringa video
            stringa_log_schermo += f"Rimbalzo {bounce+1}: Triangolo {idx_faccia} -> XYZ: {np.round(punto_contatto, 1)}\n"

            # Disegna il raggio corrente e la sfera nel punto di impatto
            attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_contatto), color=colori_raggi[bounce], line_width=4))
            attori_raggi.append(plotter.add_mesh(pv.PolyData(punto_contatto), color=colori_raggi[bounce], point_size=12, render_points_as_spheres=True))

            # Calcolo della riflessione specchiata tramite normali Trimesh
            normale_grezza = mesh_trimesh.face_normals[idx_faccia].copy()
            normale_grezza /= np.linalg.norm(normale_grezza)
            
            normale_superficie = -normale_grezza if np.dot(direzione_corrente, normale_grezza) > 0 else normale_grezza
            
            direzione_riflessa = direzione_corrente - 2 * np.dot(direzione_corrente, normale_superficie) * normale_superficie
            direzione_riflessa /= np.linalg.norm(direzione_riflessa)

            # Aggiornamento parametri per il ciclo successivo con offset epsilon centimetrico
            direzione_corrente = direzione_riflessa
            origine_corrente = punto_contatto + (normale_superficie * 1e-2)
        else:
            # Il raggio si disperde nel cielo aperto
            punto_fine_disperso = origine_corrente + direzione_corrente * 1000.0
            attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_fine_disperso), color=colori_raggi[bounce], line_width=2))
            stringa_log_schermo += f"Rimbalzo {bounce+1}: Disperso nello spazio.\n"
            break

    # 4. AGGIORNA IL PANNELLO DI TESTO SOVRIMPRESSO IN ALTO A DESTRA (In tempo reale)
    attore_testo = plotter.add_text(
        stringa_log_schermo, 
        position="upper_right", 
        font_size=10, 
        color="white", 
        font="courier",
        shadow=True
    )


# --- 4. FUNZIONI DI CALLBACK PER AGGIORNARE LO STATO ---
def cb_xx(val): stato_parametri["xx"] = val; aggiorna_ray_tracing()
def cb_yy(val): stato_parametri["yy"] = val; aggiorna_ray_tracing()
def cb_zz(val): stato_parametri["zz"] = val; aggiorna_ray_tracing()
def cb_az(val): stato_parametri["azimut"] = val; aggiorna_ray_tracing()
def cb_el(val): stato_parametri["elevazione"] = val; aggiorna_ray_tracing()


# --- 5. CREAZIONE DEI 5 SLIDER INTERATTIVI ---
plotter.add_slider_widget(callback=cb_xx, rng=[xmin-offset_pos, xmax+offset_pos], value=stato_parametri["xx"], title="Posizione X Sorgente", pointa=(0.05, 0.92), pointb=(0.30, 0.92), style="classic", color="black")
plotter.add_slider_widget(callback=cb_yy, rng=[ymin-offset_pos, ymax+offset_pos], value=stato_parametri["yy"], title="Posizione Y Sorgente", pointa=(0.05, 0.81), pointb=(0.30, 0.81), style="classic", color="black")
plotter.add_slider_widget(callback=cb_zz, rng=[zmin, zmax+offset_pos], value=stato_parametri["zz"], title="Posizione Z (Altezza)", pointa=(0.05, 0.70), pointb=(0.30, 0.70), style="classic", color="black")
plotter.add_slider_widget(callback=cb_az, rng=[0.0, 360.0], value=stato_parametri["azimut"], title="Azimut (Gradi)", pointa=(0.05, 0.59), pointb=(0.30, 0.59), style="classic", color="black")
plotter.add_slider_widget(callback=cb_el, rng=[-90.0, 90.0], value=stato_parametri["elevazione"], title="Elevazione (Gradi)", pointa=(0.05, 0.48), pointb=(0.30, 0.48), style="classic", color="black", interaction_event="always")

# Esecuzione del calcolo iniziale al boot
aggiorna_ray_tracing()

# Avvia la visualizzazione grafica
plotter.show()


In [ ]:
# --- 1. CONFIGURAZIONE MESH ---
vertices = np.vstack((X_up, Y_up, Z_up)).T
mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=triangoli_up)
mesh_trimesh.fix_normals()

faces_pv = np.hstack(np.pad(triangoli_up, ((0, 0), (1, 0)), constant_values=3))
mesh_pv = pv.PolyData(mesh_trimesh.vertices, faces_pv)

# --- 2. CONFIGURAZIONE INIZIALE FINESTRA E TERRENO ---
plotter = pv.Plotter()

# Render della mesh in grigio uniforme, senza mappe di colore o colorbar
plotter.add_mesh(mesh_pv, color="darkgray", lighting=True, opacity=0.8)

# Calcolo dei confini e del centro geometrico del terreno
xmin, ymin, zmin = mesh_trimesh.bounds[0]
xmax, ymax, zmax = mesh_trimesh.bounds[1]
centro_x = (xmin + xmax) / 2
centro_y = (ymin + ymax) / 2

# Dizionario di stato per i parametri degli slider PyVista
stato_parametri = {
    "n_tx_lato": 10,
    "spaziatura_tx": 150.0,
    "altezza_flotta": 200.0,
    "azimut": 93.0,
    "elevazione": -45.0
}

# Liste e variabili globali di gestione della scena
attori_raggi = []
attore_testo = None
MAX_BOUNCES = 4
colori_raggi = ["orange", "green", "magenta", "cyan"]

# --- 3. GENERAZIONE DELLA FLOTTA DI TX ---
def genera_griglia_tx_spaziata(num_tx_lato, spaziatura_metri, altitudine_assoluta):
    """Genera una griglia regolare di TX centrata rispetto al terreno."""
    if num_tx_lato == 1:
        return np.array([[centro_x, centro_y, altitudine_assoluta]])
        
    estensione_asse = (num_tx_lato - 1) * spaziatura_metri
    inizio_x = centro_x - (estensione_asse / 2)
    inizio_y = centro_y - (estensione_asse / 2)
    
    x_coords = np.arange(num_tx_lato) * spaziatura_metri + inizio_x
    y_coords = np.arange(num_tx_lato) * spaziatura_metri + inizio_y
    
    X, Y = np.meshgrid(x_coords, y_coords)
    punti_2d = np.vstack([X.ravel(), Y.ravel()]).T
    quote_z = np.ones((len(punti_2d), 1)) * altitudine_assoluta
    return np.hstack([punti_2d, quote_z])

# --- 4. FUNZIONE DI AGGIORNAMENTO GEOMETRIA E TESTO (CALLBACK) ---
def aggiorna_ray_tracing(*args):
    """Aggiorna i raggi di tutti i TX della flotta e riscrive il testo a schermo."""
    global attori_raggi, attore_testo
    
    # Svuota i vecchi oggetti grafici dalla GPU
    for attore in attori_raggi:
        plotter.remove_actor(attore)
    attori_raggi.clear()
    
    if attore_testo is not None:
        plotter.remove_actor(attore_testo)

    # Genera la griglia dei TX
    quota_assoluta_volo = zmax + stato_parametri["altezza_flotta"]
    lista_trasmettitori = genera_griglia_tx_spaziata(
        int(stato_parametri["n_tx_lato"]), 
        stato_parametri["spaziatura_tx"], 
        quota_assoluta_volo
    )
    
    direzione_iniziale = calcola_direzione_raggio(stato_parametri["azimut"], stato_parametri["elevazione"])
    
    stringa_log_schermo = (
        f"FLOTTA TRASMETTITORI (Tot: {len(lista_trasmettitori)})\n"
        f"Config: {int(stato_parametri['n_tx_lato'])}x{int(stato_parametri['n_tx_lato'])} | Spaziatura: {stato_parametri['spaziatura_tx']:.0f}m\n"
        f"Angoli: Az={stato_parametri['azimut']:.1f}°, El={stato_parametri['elevazione']:.1f}°\n"
        f"--------------------------------------------------\n"
    )

    # Ciclo di calcolo indipendente per ogni TX della flotta
    for idx_tx, tx_fittizio in enumerate(lista_trasmettitori):
        # Disegna il trasmettitore corrente come una sfera rossa
        tx_sfera = pv.Sphere(radius=25.0, center=tx_fittizio)
        attori_raggi.append(plotter.add_mesh(tx_sfera, color="red", lighting=True))
        
        origine_corrente = tx_fittizio.copy()
        direzione_corrente = direzione_iniziale.copy()
        
        # Sotto-ciclo ricorsivo di multi-rimbalzo
        for bounce in range(MAX_BOUNCES):
            p_end = origine_corrente + (direzione_corrente * 10000.0)
            p_intersezioni, id_facce = mesh_pv.ray_trace(origine_corrente, p_end)

            if len(p_intersezioni) > 0:
                # Ordinamento per vicinanza spaziale
                distanze = np.linalg.norm(p_intersezioni - origine_corrente, axis=1)
                idx_vicino = np.argmin(distanze)
                punto_contatto = p_intersezioni[idx_vicino, :]
                
                if hasattr(id_facce, "__len__"):
                    idx_faccia = int(id_facce[idx_vicino]) if len(id_facce) > 0 else int(id_facce)
                else:
                    idx_faccia = int(id_facce)

                # Memorizza nel log di testo l'impatto del raggio corrente
                if idx_tx < 4:  # Limita le righe di testo visibili a schermo per evitare overflow grafico
                    stringa_log_schermo += f"TX #{idx_tx+1} - Rimb.{bounce+1}: T_{idx_faccia} -> XYZ: {np.round(punto_contatto, 1)}\n"

                # Disegna il vettore del raggio e il marcatore di impatto asferico
                attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_contatto), color=colori_raggi[bounce], line_width=3))
                
                dim_sfera = max(2.0, 12.0 - (bounce * 2))
                attori_raggi.append(plotter.add_mesh(pv.Sphere(radius=dim_sfera, center=punto_contatto), color=colori_raggi[bounce]))

                # Calcolo geometrico del rimbalzo specchiato
                normale_grezza = mesh_trimesh.face_normals[idx_faccia].copy()
                normale_grezza /= np.linalg.norm(normale_grezza)
                normale_superficie = -normale_grezza if np.dot(direzione_corrente, normale_grezza) > 0 else normale_grezza
                
                direzione_riflessa = direzione_corrente - 2 * np.dot(direzione_corrente, normale_superficie) * normale_superficie
                direzione_riflessa /= np.linalg.norm(direzione_riflessa)

                # Aggiorna coordinate applicando l'offset epsilon antiautointersezione
                direzione_corrente = direzione_riflessa
                origine_corrente = punto_contatto + (normale_superficie * 1e-2)
            else:
                # Il raggio si disperde nel cielo aperto
                punto_fine_disperso = origine_corrente + direzione_corrente * 1000.0
                attori_raggi.append(plotter.add_mesh(pv.Line(origine_corrente, punto_fine_disperso), color=colori_raggi[bounce], line_width=1.5))
                break

    # Sovraimprime il pannello informativo dinamico in alto a destra
    attore_testo = plotter.add_text(stringa_log_schermo, position="upper_right", font_size=9, color="white", font="courier", shadow=True)

# --- 5. FUNZIONI DI CALLBACK PER AGGIORNARE LO STATO ---
def cb_n_tx(val): stato_parametri["n_tx_lato"] = int(val); aggiorna_ray_tracing()
def cb_spaz(val): stato_parametri["spaziatura_tx"] = val; aggiorna_ray_tracing()
def cb_alt(val): stato_parametri["altezza_flotta"] = val; aggiorna_ray_tracing()
def cb_az(val): stato_parametri["azimut"] = val; aggiorna_ray_tracing()
def cb_el(val): stato_parametri["elevazione"] = val; aggiorna_ray_tracing()

# --- 6. AGGIUNTA DEI 5 SLIDER INTERATTIVI NELL'INTERFACCIA PYVISTA ---
plotter.add_slider_widget(callback=cb_n_tx, rng=[1.0, 10.0], value=stato_parametri["n_tx_lato"], title="TX per lato", pointa=(0.05, 0.92), pointb=(0.30, 0.92), style="classic", color="black", fmt="%1.0f")
plotter.add_slider_widget(callback=cb_spaz, rng=[10.0, 500.0], value=stato_parametri["spaziatura_tx"], title="Spaziatura TX (m)", pointa=(0.05, 0.81), pointb=(0.30, 0.81), style="classic", color="black")
plotter.add_slider_widget(callback=cb_alt, rng=[50.0, 1000.0], value=stato_parametri["altezza_flotta"], title="Altezza Flotta TX", pointa=(0.05, 0.70), pointb=(0.30, 0.70), style="classic", color="black")
plotter.add_slider_widget(callback=cb_az, rng=[0.0, 360.0], value=stato_parametri["azimut"], title="Azimut (Gradi)", pointa=(0.05, 0.59), pointb=(0.30, 0.59), style="classic", color="black")
plotter.add_slider_widget(callback=cb_el, rng=[-90.0, 90.0], value=stato_parametri["elevazione"], title="Elevazione (Gradi)", pointa=(0.05, 0.48), pointb=(0.30, 0.48), style="classic", color="black")

# Esecuzione del rendering iniziale al boot
aggiorna_ray_tracing()
plotter.show()

print("WOW")